# NLP Lab Assignment 1: Regex Pipelines
## Question 1: Financial Transaction Pipeline

### Scenario: Processing system logs to remove HTML/metadata, tokenize identifiers (preserving underscores), extract Transaction IDs (TXN-XXXX), and validate payment status.

In [15]:
import re
import pandas as pd


In [16]:
# Data Samples for Question 1
q1_samples = [
    "<div>System_Log: User @Ali processed payment for 'Premium_Sub' on 2026-02-18. ID: TXN-5582. Status: Paid!</div>",
    "<div>System_Log: User @Zain attempted 'Basic_Plan' purchase. ID: TXN-9910. Error: Funds_Low. Status: Unpaid.</div>",
    "<div>System_Log: [IP: 192.168.1.1] User @Sana login successful. ID: TXN-2241. Status: Paid. Reference: http://logs.pucit.edu.pk/txn2241</div>",
    "<div>System_Log: User @Hamza requested refund for ID: TXN-4403. Verification: Pending. Status: Unpaid.</div>",
    "<div>System_Log: TXN-8827 confirmed for @User_99. Amount: 5000PKR. Status: Paid. (Verified)</div>"
]

def process_q1(samples):
    results = []
    for i, raw_text in enumerate(samples, 1):
        # --- STEP 1: Data Cleaning ---
        # Regex Explanation:
        # </?div>        -> Matches <div> or </div>
        # System_Log:\s* -> Matches prefix and optional space
        # \[IP:.*?\]\s* -> Matches bracketed IP metadata
        # http\S+         -> Matches URLs
        clean_pattern = r"<(?:/)?div>|System_Log:\s*|\[IP:.*?\]\s*|http\S+"
        cleaned = re.sub(clean_pattern, "", raw_text).strip()

        # --- STEP 2: Tokenization ---
        # \b\w+(?:_\w+)*\b -> Matches words and preserves underscores (e.g., Premium_Sub)
        tokens = re.findall(r"\b\w+(?:_\w+)*\b", cleaned)

        # --- STEP 3: Information Extraction ---
        # TXN-\d{4} -> Matches literal 'TXN-' followed by 4 digits
        txn_match = re.search(r"TXN-\d{4}", cleaned)
        extracted = txn_match.group(0) if txn_match else "None"

        # --- STEP 4: Validation ---
        # Logic: Determine status based on presence of 'Paid' token
        status = "Paid" if "Paid" in tokens else "Unpaid"

        results.append({"Sample": f"Sample {i}", "Cleaning": cleaned, "Tokenization": str(tokens), "Extraction": extracted, "Validation": status})
    return results

# Execute and Display
q1_res = process_q1(q1_samples)
for res in q1_res:
    print(f"--- {res['Sample']} ---\nCleaning: {res['Cleaning']}\nTokenization: {res['Tokenization']}\nExtraction: {res['Extraction']}\nValidation: {res['Validation']}\n")

print("=== TABLE OUTPUT FOR SAMPLE 1 ===")
display(pd.DataFrame([{"Step": k, "Expected Output": q1_res[0][k]} for k in ["Cleaning", "Tokenization", "Extraction", "Validation"]]))

--- Sample 1 ---
Cleaning: User @Ali processed payment for 'Premium_Sub' on 2026-02-18. ID: TXN-5582. Status: Paid!
Tokenization: ['User', 'Ali', 'processed', 'payment', 'for', 'Premium_Sub', 'on', '2026', '02', '18', 'ID', 'TXN', '5582', 'Status', 'Paid']
Extraction: TXN-5582
Validation: Paid

--- Sample 2 ---
Cleaning: User @Zain attempted 'Basic_Plan' purchase. ID: TXN-9910. Error: Funds_Low. Status: Unpaid.
Tokenization: ['User', 'Zain', 'attempted', 'Basic_Plan', 'purchase', 'ID', 'TXN', '9910', 'Error', 'Funds_Low', 'Status', 'Unpaid']
Extraction: TXN-9910
Validation: Unpaid

--- Sample 3 ---
Cleaning: User @Sana login successful. ID: TXN-2241. Status: Paid. Reference:
Tokenization: ['User', 'Sana', 'login', 'successful', 'ID', 'TXN', '2241', 'Status', 'Paid', 'Reference']
Extraction: TXN-2241
Validation: Paid

--- Sample 4 ---
Cleaning: User @Hamza requested refund for ID: TXN-4403. Verification: Pending. Status: Unpaid.
Tokenization: ['User', 'Hamza', 'requested', 'refund', 'fo

,Step,Expected Output
0,Cleaning,User @Ali processed payment for 'Premium_Sub' ...
1,Tokenization,"['User', 'Ali', 'processed', 'payment', 'for',..."
2,Extraction,TXN-5582
3,Validation,Paid


## Question 2: Social Media Monitoring

### Scenario: Monitoring event mentions to remove mentions/URLs, tokenize hashtags, extract dates (YYYY-MM-DD), and infer sentiment.

In [19]:
# Data Samples for Question 2
q2_samples = [
    "<div>@Ali: Can't wait for the NLP_Summit on 2026-05-12! #Excited. Details at http://event.ai/join</div>",
    "<div>@Zain: Thinking about the workshop on 2026-06-15. #Interested. http://link.com</div>",
    "<div>@Sana: I will attend the 2026-07-20 meeting. #Neutral. No link.</div>",
    "<div>@Hamza: 2026-08-05 is the big day! #Excited to see you all. http://link.ai</div>",
    "<div>@Admin: Checking the 2026-09-10 calendar. #Interested. http://pucit.edu.pk</div>"
]

def process_q2(samples):
    results = []
    for i, raw_text in enumerate(samples, 1):
        # --- STEP 1: Data Cleaning ---
        # @\w+:?\s* -> Removes @mentions and trailing colons/spaces
        clean_pattern = r"<(?:/)?div>|@\w+:?\s*|http\S+"
        cleaned = re.sub(clean_pattern, "", raw_text).strip()

        # --- STEP 2: Tokenization ---
        # #\w+       -> Captures hashtags specifically
        # \b\w+...   -> Captures standard words and underscores
        tokens = re.findall(r"#\w+|\b\w+(?:_\w+)*\b", cleaned)

        # --- STEP 3: Information Extraction ---
        # \d{4}-\d{2}-\d{2} -> Standard YYYY-MM-DD date format
        date_match = re.search(r"\d{4}-\d{2}-\d{2}", cleaned)
        extracted = date_match.group(0) if date_match else "None"

        # --- STEP 4: Inference ---
        # Extract sentiment from the specific hashtag used
        sentiment = next((t.replace("#", "") for t in tokens if t in ["#Excited", "#Interested", "#Neutral"]), "None")

        results.append({"Sample": f"Sample {i}", "Cleaning": cleaned, "Tokenization": str(tokens), "Extraction": extracted, "Inference": sentiment})
    return results

q2_res = process_q2(q2_samples)
for res in q2_res:
    print(f"--- {res['Sample']} ---\nCleaning: {res['Cleaning']}\nTokenization: {res['Tokenization']}\nExtraction: {res['Extraction']}\nInference: {res['Inference']}\n")

print("=== TABLE OUTPUT FOR SAMPLE 1 ===")
display(pd.DataFrame([{"Step": k, "Expected Output": q2_res[0][k]} for k in ["Cleaning", "Tokenization", "Extraction", "Inference"]]))

--- Sample 1 ---
Cleaning: Can't wait for the NLP_Summit on 2026-05-12! #Excited. Details at
Tokenization: ['Can', 't', 'wait', 'for', 'the', 'NLP_Summit', 'on', '2026', '05', '12', '#Excited', 'Details', 'at']
Extraction: 2026-05-12
Inference: Excited

--- Sample 2 ---
Cleaning: Thinking about the workshop on 2026-06-15. #Interested.
Tokenization: ['Thinking', 'about', 'the', 'workshop', 'on', '2026', '06', '15', '#Interested']
Extraction: 2026-06-15
Inference: Interested

--- Sample 3 ---
Cleaning: I will attend the 2026-07-20 meeting. #Neutral. No link.
Tokenization: ['I', 'will', 'attend', 'the', '2026', '07', '20', 'meeting', '#Neutral', 'No', 'link']
Extraction: 2026-07-20
Inference: Neutral

--- Sample 4 ---
Cleaning: 2026-08-05 is the big day! #Excited to see you all.
Tokenization: ['2026', '08', '05', 'is', 'the', 'big', 'day', '#Excited', 'to', 'see', 'you', 'all']
Extraction: 2026-08-05
Inference: Excited

--- Sample 5 ---
Cleaning: Checking the 2026-09-10 calendar. #Interes

,Step,Expected Output
0,Cleaning,Can't wait for the NLP_Summit on 2026-05-12! #...
1,Tokenization,"['Can', 't', 'wait', 'for', 'the', 'NLP_Summit..."
2,Extraction,2026-05-12
3,Inference,Excited


## Question 3: Security Clearance Auditing

### Scenario: Auditing server room logs to remove timestamps, tokenize identifiers, extract Badge IDs (BID-XXXXX), and validate access.

In [20]:
# Data Samples for Question 3
q3_samples = [
    "<div>[08:30:15] LOG: User_A swiped at Entry_1. ID: BID-10293. Status: Authorized.</div>",
    "<div>[09:45:00] LOG: User_B attempted restricted area. ID: BID-44921. Status: Denied. Access Level: Low.</div>",
    "<div>[10:15:22] LOG: Guest_X check-in complete. ID: BID-00211. Status: Authorized. http://security.pucit.edu.pk/logs</div>",
    "<div>[12:00:10] LOG: User_C failed verification. ID: BID-99382. Status: Denied.</div>",
    "<div>[14:20:55] LOG: Maintenance_Z cleared for entry. ID: BID-33810. Status: Authorized. (Verified_Biometrically)</div>"
]

def process_q3(samples):
    results = []
    for i, raw_text in enumerate(samples, 1):
        # --- STEP 1: Data Cleaning ---
        # \[\d{2}:\d{2}:\d{2}\] -> Removes bracketed time (HH:MM:SS)
        # LOG:\s* -> Removes log prefix
        clean_pattern = r"<(?:/)?div>|\[\d{2}:\d{2}:\d{2}\]\s*|LOG:\s*|http\S+"
        cleaned = re.sub(clean_pattern, "", raw_text).strip()

        # --- STEP 2: Tokenization ---
        tokens = re.findall(r"\b\w+(?:_\w+)*\b", cleaned)

        # --- STEP 3: Information Extraction ---
        # BID-\d{5} -> Matches literal 'BID-' followed by exactly 5 digits
        bid_match = re.search(r"BID-\d{5}", cleaned)
        extracted = bid_match.group(0) if bid_match else "None"

        # --- STEP 4: Validation ---
        status = "Authorized" if "Authorized" in tokens else "Denied"

        results.append({"Sample": f"Sample {i}", "Cleaning": cleaned, "Tokenization": str(tokens), "Extraction": extracted, "Validation": status})
    return results

q3_res = process_q3(q3_samples)
for res in q3_res:
    print(f"--- {res['Sample']} ---\nCleaning: {res['Cleaning']}\nTokenization: {res['Tokenization']}\nExtraction: {res['Extraction']}\nValidation: {res['Validation']}\n")

print("=== TABLE OUTPUT FOR SAMPLE 1 ===")
display(pd.DataFrame([{"Step": k, "Expected Output": q3_res[0][k]} for k in ["Cleaning", "Tokenization", "Extraction", "Validation"]]))

--- Sample 1 ---
Cleaning: User_A swiped at Entry_1. ID: BID-10293. Status: Authorized.
Tokenization: ['User_A', 'swiped', 'at', 'Entry_1', 'ID', 'BID', '10293', 'Status', 'Authorized']
Extraction: BID-10293
Validation: Authorized

--- Sample 2 ---
Cleaning: User_B attempted restricted area. ID: BID-44921. Status: Denied. Access Level: Low.
Tokenization: ['User_B', 'attempted', 'restricted', 'area', 'ID', 'BID', '44921', 'Status', 'Denied', 'Access', 'Level', 'Low']
Extraction: BID-44921
Validation: Denied

--- Sample 3 ---
Cleaning: Guest_X check-in complete. ID: BID-00211. Status: Authorized.
Tokenization: ['Guest_X', 'check', 'in', 'complete', 'ID', 'BID', '00211', 'Status', 'Authorized']
Extraction: BID-00211
Validation: Authorized

--- Sample 4 ---
Cleaning: User_C failed verification. ID: BID-99382. Status: Denied.
Tokenization: ['User_C', 'failed', 'verification', 'ID', 'BID', '99382', 'Status', 'Denied']
Extraction: BID-99382
Validation: Denied

--- Sample 5 ---
Cleaning: Maint

,Step,Expected Output
0,Cleaning,User_A swiped at Entry_1. ID: BID-10293. Statu...
1,Tokenization,"['User_A', 'swiped', 'at', 'Entry_1', 'ID', 'B..."
2,Extraction,BID-10293
3,Validation,Authorized
